In [12]:
pip install pandas numpy scikit-learn xgboost catboost statsmodels openpyxl

Note: you may need to restart the kernel to use updated packages.


In [14]:
# =====================================================================
# 01 - VERİ HAZIRLAMA
# all_data24.csv (uzun format) → geniş format
# Çıktı: model_giris.csv  (il, yıl × uzman, pratisyen, asistan, hemsire,
#                          ebe, diger, hastane, yatak, nufus)
# =====================================================================

import pandas as pd
import numpy as np

GIRIS = "all_data24.csv"          # gerek varsa dosya yolunu düzenle
CIKTI = "model_giris.csv"

In [16]:
# 1) Veriyi oku ------------------------------------------------------
df = pd.read_csv(GIRIS)
print("Ham veri boyutu:", df.shape)
print("Sütunlar:", df.columns.tolist())

Ham veri boyutu: (57753, 6)
Sütunlar: ['il_adı', 'yıl', 'değer', 'veri_tipi', 'kategori', 'kurum']


In [18]:
# 2) Eczacı ve dişçiyi at -------------------------------------------
df = df[~df["kategori"].isin(["eczaci", "dis"])].copy()

In [20]:
# 3) Hastane: 3 kurum (devlet/özel/üniversite) toplamı --------------
hastane = (df[df["veri_tipi"] == "hastane_sayisi"]
           .groupby(["il_adı", "yıl"])["değer"].sum().reset_index()
           .rename(columns={"değer": "hastane"}))

In [22]:
# 4) Yatak: 3 kurum toplamı -----------------------------------------
yatak = (df[df["veri_tipi"] == "yatak_sayisi"]
         .groupby(["il_adı", "yıl"])["değer"].sum().reset_index()
         .rename(columns={"değer": "yatak"}))

In [24]:
# 5) Nüfus ----------------------------------------------------------
nufus = (df[df["veri_tipi"] == "nufus"]
         .groupby(["il_adı", "yıl"])["değer"].sum().reset_index()
         .rename(columns={"değer": "nufus"}))

In [26]:
# 6) Personel: her kategori (uzman/pratisyen/asistan/hemsire/ebe/diger)
personel_kats = ["uzman", "pratisyen", "asistan", "hemsire", "ebe", "diger"]
personel_pivot = (df[df["kategori"].isin(personel_kats)]
                  .groupby(["il_adı", "yıl", "kategori"])["değer"].sum()
                  .reset_index()
                  .pivot(index=["il_adı", "yıl"], columns="kategori", values="değer")
                  .reset_index())
personel_pivot.columns.name = None

In [28]:
# 7) Hepsini birleştir ----------------------------------------------
veri = (hastane
        .merge(yatak,         on=["il_adı", "yıl"], how="outer")
        .merge(nufus,         on=["il_adı", "yıl"], how="outer")
        .merge(personel_pivot, on=["il_adı", "yıl"], how="outer"))

# Eksik (NaN) hücreleri il bazında zaman yönünde lineer interpolasyon
veri = veri.sort_values(["il_adı", "yıl"]).reset_index(drop=True)
say_kolon = ["hastane", "yatak", "nufus"] + personel_kats
veri[say_kolon] = (veri.groupby("il_adı")[say_kolon]
                       .apply(lambda x: x.interpolate(limit_direction="both"))
                       .reset_index(drop=True))

veri = veri.dropna(subset=["nufus"]).copy()

In [30]:
print("\nGeniş format boyutu:", veri.shape)
print("Yıl aralığı:", veri["yıl"].min(), "-", veri["yıl"].max())
print("İl sayısı:", veri["il_adı"].nunique())
print(veri.head())

veri.to_csv(CIKTI, index=False, encoding="utf-8-sig")
print(f"\n[OK] {CIKTI} kaydedildi.")


Geniş format boyutu: (1863, 11)
Yıl aralığı: 2002 - 2024
İl sayısı: 81
  il_adı   yıl  hastane   yatak      nufus  asistan   diger     ebe  hemsire  \
0  Adana  2002     18.0  4266.0  1916637.0    646.0  1182.0  1110.0   1927.0   
1  Adana  2003     21.0  4382.0  1933428.0    794.0  1414.0  1147.0   2118.0   
2  Adana  2004     21.0  4207.0  1951142.0    856.0  1469.0  1167.0   2157.0   
3  Adana  2005     22.0  4773.0  1969512.0    869.0  1540.0  1186.0   2223.0   
4  Adana  2006     21.0  4973.0  1988277.0    409.0  1683.0  1258.0   2235.0   

   pratisyen   uzman  
0      826.0  1227.0  
1      844.0  1439.0  
2      892.0  1497.0  
3      979.0  1555.0  
4      938.0  1364.0  

[OK] model_giris.csv kaydedildi.
